# Lab 05-03: Scope State and Add Long-Term Memory

**Book chapter:** Chapter 05: Models and Building Agents With ADK  
**Estimated time:** 30 minutes

## Overview

Cymbal Retail's personal shopping agent should remember a shopper's shoe size in every
conversation, remember the store only for the current conversation, and recall facts
from earlier conversations, such as an upcoming hiking trip. In this lab, you store
values with different state scopes, add long-term memory with `preload_memory` and an
after-agent callback, and then switch the same agent to Memory Bank.

## Learning Objectives

By the end of this lab, you will be able to:

- Choose the right state scope: no prefix, user:, or temp:
- Read state values in instructions with placeholders
- Save conversations to memory with an after-agent callback
- Retrieve memories automatically with preload_memory
- Use Memory Bank as a managed long-term memory service

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** Less than USD 1 if you delete the Memory Bank instance in Step 6.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Need | Where to store it |
|---|---|
| A value for one conversation | State with no prefix |
| A value for all of one user's conversations | State with the `user:` prefix |
| A value for everyone | State with the `app:` prefix |
| A value for one invocation only | State with the `temp:` prefix |
| Facts extracted and searched across conversations | Memory Bank |

## Setup

In [ ]:
%pip install --quiet --upgrade google-adk google-cloud-aiplatform[agent_engines]

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Configure ADK and create helpers

The `chat` helper runs an agent for a given user and session, optionally with a memory
service. `state_of` returns a session's state so you can see what each scope stores.
`REGION` is the region for the Memory Bank instance in Step 5.

In [ ]:
import asyncio
import os
import time
from concurrent.futures import ThreadPoolExecutor

import vertexai
from google.adk.agents import Agent
from google.adk.memory import InMemoryMemoryService
from google.adk.memory import VertexAiMemoryBankService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import ToolContext, preload_memory
from google.genai import types

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
REGION = "us-central1"  # The region for the Memory Bank instance
sessions = InMemorySessionService()


def chat(agent, user_id, session_id, text, memory=None):
    """Sends one message and prints the agent's final answer."""
    runner = Runner(app_name="cymbal", agent=agent, memory_service=memory,
                    session_service=sessions, auto_create_session=True)
    message = types.Content(role="user", parts=[types.Part(text=text)])
    for event in runner.run(user_id=user_id, session_id=session_id,
                            new_message=message):
        if event.is_final_response() and event.content and \
                event.content.parts:
            print(f"\n[{user_id}/{session_id}] {text}")
            print("  Agent:", event.content.parts[0].text)


def state_of(user_id, session_id):
    """Returns a copy of a session's state."""
    lookup = sessions.get_session(app_name="cymbal", user_id=user_id,
                                  session_id=session_id)
    with ThreadPoolExecutor(1) as pool:  # Works in scripts and notebooks.
        return dict(pool.submit(asyncio.run, lookup).result().state)

## Step 2: Store values with three different state scopes

The `save_preferences` tool writes three values with different scopes:

| Key | Scope |
|---|---|
| `user:shoe_size` | Every session of this user |
| `store` | This session only |
| `temp:raw_size` | This invocation only; never saved |

The instruction reads `{user:shoe_size?}` and `{store?}`. The `?` means the placeholder
is left empty instead of failing when the value does not exist yet.

In [ ]:
def save_preferences(shoe_size: str, store: str,
                     tool_context: ToolContext) -> dict:
    """Saves the shopper's shoe size and the store they are asking about.

    Args:
        shoe_size: The shopper's shoe size, for example 10.
        store: The store location mentioned in this conversation.
    """
    tool_context.state["user:shoe_size"] = shoe_size  # All user sessions
    tool_context.state["store"] = store  # This session only
    tool_context.state["temp:raw_size"] = shoe_size  # This invocation only
    return {"saved": True}


stylist = Agent(
    name="stylist",
    model=MODEL,
    instruction="You recommend Cymbal Retail boots. The shopper's shoe "
                "size is {user:shoe_size?} and their store is {store?}. "
                "When the shopper mentions a size or a store, call "
                "save_preferences. Mention the size in recommendations "
                "whenever you know it.",
    tools=[save_preferences],
)
chat(stylist, "maria", "session-1",
     "I wear size 10 and I shop at the Denver store.")
print("State of session-1:", state_of("maria", "session-1"))

**Expected result:** the state contains `store` and `user:shoe_size`, but not `temp:raw_size`.

## Step 3: Start new sessions and compare what each one can see

Start a new session for the same shopper, and a session for a different shopper.

In [ ]:
chat(stylist, "maria", "session-2", "Which boots would you recommend?")
print("State of session-2:", state_of("maria", "session-2"))
chat(stylist, "james", "session-3", "Which boots would you recommend?")
print("State of James's session:", state_of("james", "session-3"))

**Expected result:** Maria's second session sees `user:shoe_size` but not `store`, and the recommendation mentions size 10. James's session sees neither.

## Step 4: Add long-term memory with an in-memory memory service

State holds values that code writes explicitly. **Long-term memory** recalls facts from
earlier conversations. The `preload_memory` tool searches memory at the start of each
turn and adds relevant past conversations to the request. The after-agent callback
saves each conversation to the memory service.

In [ ]:
async def save_session(callback_context):
    """After-agent callback: sends the conversation to memory."""
    await callback_context.add_session_to_memory()


concierge = Agent(
    name="concierge",
    model=MODEL,
    instruction="You are Cymbal Retail's shopping concierge. Use what you "
                "remember about the shopper to personalize every answer.",
    tools=[preload_memory],
    after_agent_callback=save_session,
)
local_memory = InMemoryMemoryService()
chat(concierge, "maria", "trip-1", "I'm hiking the Colorado Trail in "
     "September, and my feet are wide.", local_memory)
chat(concierge, "maria", "trip-2", "Which boots should I buy for the "
     "Colorado Trail?", local_memory)

**Expected result:** in the second session, the concierge recommends wide-fit boots for the Colorado Trail without being told about the shopper's feet again.

## Step 5: Switch to Memory Bank, the managed long-term memory service

**Memory Bank** is the managed memory service. Instead of storing raw conversations, it
uses Gemini to extract and consolidate facts. The cell creates an Agent Runtime instance
that hosts Memory Bank, runs the same agent with it, waits for the extracted
memories, and then starts a new session.

> **Note:** The instance is billed while it stores memories. Run Step 6 to delete it.

In [ ]:
client = vertexai.Client(project=PROJECT, location=REGION)
# Create an Agent Runtime instance that hosts Memory Bank.
engine = client.agent_engines.create(
    config={"display_name": "cymbal-memory-bank"})
ENGINE_NAME = engine.api_resource.name
print("Created Memory Bank instance:", ENGINE_NAME)
memory_bank = VertexAiMemoryBankService(
    project=PROJECT, location=REGION,
    agent_engine_id=ENGINE_NAME.split("/")[-1])

chat(concierge, "maria", "bank-1", "I'm hiking the Colorado Trail in "
     "September, and my feet are wide.", memory_bank)
# Memory Bank extracts memories in the background, so check for a while.
for attempt in range(12):
    facts = [m.fact for m in client.agent_engines.list_memories(
        name=ENGINE_NAME)]
    if facts:
        print("Memories extracted by Memory Bank:", facts)
        break
    time.sleep(10)
chat(concierge, "maria", "bank-2", "Which boots should I buy for my "
     "hike?", memory_bank)

**Expected result:** a list of extracted facts, such as the wide feet and the September hike, and a personalized recommendation in the new session.

## Step 6: Clean up by deleting the Memory Bank instance

Delete the Memory Bank instance.

In [ ]:
client.agent_engines.delete(name=ENGINE_NAME, force=True)
print("Deleted", ENGINE_NAME)

## Check Your Understanding

**1. Where would you store a shopper's preferred language?**

<details><summary>Show answer</summary>

In state with the user: prefix, so that every conversation with that shopper can use it.

</details>

**2. Why isn't temp:raw_size in the saved state?**

<details><summary>Show answer</summary>

Values with the temp: prefix last only for the current invocation and are never saved.

</details>

**3. How does Memory Bank differ from saving whole conversations?**

<details><summary>Show answer</summary>

Memory Bank uses Gemini to extract and consolidate facts, which keeps memories concise, searchable, and up to date across conversations.

</details>

## If Something Went Wrong

If the notebook stopped before Step 6, run the next cell to delete any Memory Bank instance
that this lab created.

In [ ]:
import os

import vertexai

client = vertexai.Client(project=os.environ["GOOGLE_CLOUD_PROJECT"],
                         location="us-central1")
for engine in client.agent_engines.list():
    if engine.api_resource.display_name == "cymbal-memory-bank":
        client.agent_engines.delete(name=engine.api_resource.name,
                                    force=True)
        print("Deleted", engine.api_resource.name)
print("Cleanup complete.")

## Summary

You stored values in three state scopes and saw which sessions can read them, added
long-term memory with `preload_memory` and an after-agent callback, and used Memory Bank to
extract facts that personalize a new conversation.